**Table of contents**<a id='toc0_'></a>    
- [Implementing Causal Attention](#toc1_)    
  - [1) Softmax → Mask → Renormalize (Technical Way)](#toc1_1_)    
    - [Step 1](#toc1_1_1_)    
    - [Step 2](#toc1_1_2_)    
    - [Step 3](#toc1_1_3_)    
  - [2) Mask (−∞) → Softmax (Real Implementation)](#toc1_2_)    
  - [Masking additional attention weights with dropout](#toc1_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Implementing Causal Attention](#toc0_)

In this section, we modify the standard self-attention mechanism to create a causal
attention mechanism, which is essential for developing an LLM in the subsequent chapters.

Causal attention, also known as masked attention, is a specialized form of self-attention.


It restricts a model to only consider previous and current inputs in a sequence when
processing any given token. This is in contrast to the standard self-attention mechanism, which allows access to the entire input sequence at once.


Consequently, when computing attention scores, the causal attention mechanism
ensures that the model only factors in tokens that occur at or before the current token in the sequence.


To achieve this in GPT-like LLMs, for each token processed, we mask out the future
tokens, which come after the current token in the input text

`Reference: Build an LLM from Scratch by Sebastian Raschka, 87-88`

## <a id='toc1_1_'></a>[1) Softmax → Mask → Renormalize (Technical Way)](#toc0_)

- Initial State: Attention scores (unnormalized)

- Step 1: Apply softmax

    This transitions the attention scores to Attention weights (normalized).

    Note: "Normalized" means that the values in each row sum to 1.

- Step 2: Mask with 0's above diagonal

    This transitions the attention weights to Masked attention scores (unnormalized).

- Step 3: Normalize rows

    This final step transitions the scores to the final Masked attention weights (normalized).

In [3]:
import numpy as npy 
import torch 
from torch.nn import Linear, Module

In [19]:
torch.manual_seed(123)

In [20]:
class SelfAttentionV2(Module): 
    def __init__(self, d_in, d_out): 
        super().__init__()
        # Initialize the weights
        self.d_out = d_out 
        self.W_query = Linear(d_in, d_out, bias=False)
        self.W_key = Linear(d_in, d_out, bias=False) 
        self.W_value = Linear(d_in, d_out, bias=False)


    def forward(self , x): 
        queries = self.W_query(x) 
        keys = self.W_key(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(
            attention_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attention_weights @ values 
        return context_vec

In [22]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your  (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)
d_in = inputs.shape[-1]
d_out = 2

sa_v2 = SelfAttentionV2(d_in, d_out)
print(sa_v2(inputs))

tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)


### <a id='toc1_1_1_'></a>[Step 1](#toc0_)

In [23]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


### <a id='toc1_1_2_'></a>[Step 2](#toc0_)

In [25]:
context_length = attn_scores.shape[0]
mask_simple =  torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [35]:
masked_simple = attn_weights * mask_simple
masked_simple

tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<MulBackward0>)

### <a id='toc1_1_3_'></a>[Step 3](#toc0_)

In [39]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums 
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)


## <a id='toc1_2_'></a>[2) Mask (−∞) → Softmax (Real Implementation)](#toc0_)

In [53]:

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)

print(masked)

tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [54]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


We know implemented softmax to masked attention weights, so now we can now compute the context vector with this masked weights.

However in the next section, I cover something about causal attention mechanism that is useful for reducing the overfitting when training LLMs

## <a id='toc1_3_'></a>[Masking additional attention weights with dropout](#toc0_)

### Dropout

In [65]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [66]:
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)

In [67]:
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7160, 0.7181, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5167, 0.5147, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4101, 0.0000],
        [0.2815, 0.3430, 0.0000, 0.3434, 0.3516, 0.3368]],
       grad_fn=<MulBackward0>)


In [69]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)
batch

torch.Size([2, 6, 3])


tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])